# MOSTA Interpolated Slices


In [ ]:
from pathlib import Path
import json
import shutil
import sys


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "assets").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate cytobridge-downstream repository root.")


DOWNSTREAM_ROOT = find_repo_root()
VENDOR_ROOT = DOWNSTREAM_ROOT / "vendor"
for path in (DOWNSTREAM_ROOT, VENDOR_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from downstream_helpers import (
    MostaRunConfig,
    build_mosta_interpolation_kwargs,
    build_mosta_snapshot_variants,
    display_svg_outputs,
    list_output_files,
    load_mosta_context,
    resolve_mosta_output_dir,
)
from CytoBridge.tl import load_label_to_color, run_interpolation_workflow, save_timepoint_snapshots


In [ ]:
config = MostaRunConfig(
    output_name="mosta_interpolated_slices_notebook",
    piecewise_spatial_warp=True,
    skip_export=True,
    skip_snapshots=False,
    classifier_cache_path="assets/mosta/classifier_cache/classifier_resmlp_52fb7dc647bfe334.pt",
)
context = load_mosta_context()
output_dir = resolve_mosta_output_dir(config)
output_dir


In [ ]:
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

interpolation = run_interpolation_workflow(
    **build_mosta_interpolation_kwargs(
        context=context,
        config=config,
        output_dir=output_dir,
    )
)

label_to_color = load_label_to_color(
    context.df[context.assets.annotation_key].astype(str).values,
    label_color_json=str(context.assets.label_color_json),
    color_h5ad=context.assets.color_h5ad,
    annotation_key=context.assets.annotation_key,
)
with open(output_dir / "label_to_color.json", "w", encoding="utf-8") as handle:
    json.dump(label_to_color, handle, indent=2)

observed_variants = build_mosta_snapshot_variants(
    context=context,
    interpolation=interpolation,
    config=config,
)

save_timepoint_snapshots(
    adata_dict=interpolation.adata_dict,
    time_keys=interpolation.time_keys,
    annotation_key=context.assets.annotation_key,
    label_to_color=label_to_color,
    observed_variants=observed_variants,
    snapshot_dir=str(output_dir / "timepoint_svg"),
    background_color=None,
    font_color="#1a1a1a",
    snapshot_point_size=2.5,
    snapshot_alpha=0.9,
    mosaic_cols=4,
    mosaic_cell_size=2.2,
    mosaic_show_title=True,
    save_pdf=True,
)

output_dir


In [ ]:
list_output_files(output_dir)


In [ ]:
svg_dir = output_dir / 'timepoint_svg'
svg_paths = [
    svg_dir / 'timepoint_mosaic.svg',
]
display_svg_outputs(svg_paths)


## Notes

- This notebook keeps snapshots and HTML outputs, but skips static `svg/pdf/png` export for speed.
- If you want the full export set, set `skip_export=False` in the config cell and rerun.
